# Checkpoint 2 — Modelagem Preditiva
## Inflação e Taxa Selic no Brasil

**Integrantes**
- Gabriel De Biasi Couto — RM 563247
- João Pedro da Silva Costa — RM 565031
- Rodrigo Campos Cordeiro — RM 566386

Este notebook continua a Parte 1 usando o mesmo conjunto de dados: IPCA (SGS 433) e Selic (SGS 1178), do Banco Central do Brasil, no recorte de janeiro de 2020 a 30/08/2026.


## 1. Problema de Machine Learning

**Objetivo:** prever a variação do **IPCA mensal do mês seguinte** a partir do comportamento recente da inflação e da taxa Selic.

**Tipo de problema:** regressão supervisionada.

**Target:** `target_ipca_proximo_mes`.

**Features:** IPCA mensal atual, IPCA acumulado em 12 meses, Selic média mensal, defasagens de 1 a 3 meses das séries, médias móveis de 3 meses e variáveis sazonais (mês em seno/cosseno).

**Relevância:** na Parte 1 os indicadores mostraram o comportamento histórico da inflação e dos juros. A modelagem preditiva adiciona uma visão prospectiva: com os dados conhecidos até um determinado mês, estimamos a inflação do mês seguinte. Isso complementa os KPIs descritivos sem afirmar causalidade entre Selic e IPCA.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))
from src.coleta_dados import preparar_base

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Coleta e pré-processamento

Etapas realizadas:
1. Coleta das mesmas séries usadas na Parte 1.
2. Agregação da Selic diária pela média mensal e integração com o IPCA pela chave `ano_mes`.
3. Cálculo do IPCA acumulado em 12 meses.
4. Criação de lags de 1, 2 e 3 meses para IPCA e Selic.
5. Criação de médias móveis de 3 meses.
6. Criação de componentes sazonais do mês (`sin` e `cos`).
7. Criação do target deslocando o IPCA mensal em -1: cada linha passa a tentar prever o mês seguinte.
8. Remoção das linhas iniciais/finais com NaN gerados pelos lags, rolling e target.
9. Separação temporal: primeiros 80% para treino e últimos 20% para teste, sem embaralhar, evitando vazamento de informação futura.


In [ ]:
ipca, selic, base = preparar_base()
base = base.sort_values('data').reset_index(drop=True)

for lag in [1, 2, 3]:
    base[f'ipca_lag{lag}'] = base['ipca_mensal'].shift(lag)
    base[f'selic_lag{lag}'] = base['selic_media_mensal'].shift(lag)

base['ipca_media3'] = base['ipca_mensal'].rolling(3).mean()
base['selic_media3'] = base['selic_media_mensal'].rolling(3).mean()
base['mes'] = base['data'].dt.month
base['mes_sin'] = np.sin(2 * np.pi * base['mes'] / 12)
base['mes_cos'] = np.cos(2 * np.pi * base['mes'] / 12)

base['target_ipca_proximo_mes'] = base['ipca_mensal'].shift(-1)

FEATURES = [
    'ipca_mensal', 'ipca_12m', 'selic_media_mensal',
    'ipca_lag1', 'ipca_lag2', 'ipca_lag3',
    'selic_lag1', 'selic_lag2', 'selic_lag3',
    'ipca_media3', 'selic_media3', 'mes_sin', 'mes_cos'
]
TARGET = 'target_ipca_proximo_mes'

modelagem = base[['data', 'ano_mes'] + FEATURES + [TARGET]].dropna().reset_index(drop=True)

print('Linhas disponíveis para modelagem:', len(modelagem))
print('Features:', len(FEATURES))
display(modelagem.head())


## 3. Separação entre treino e teste

Como os dados são temporais, não usamos `train_test_split` aleatório. A divisão mantém a ordem cronológica para que o teste represente meses posteriores aos meses usados no treinamento.


In [ ]:
corte = int(len(modelagem) * 0.80)
treino = modelagem.iloc[:corte].copy()
teste = modelagem.iloc[corte:].copy()

X_train, y_train = treino[FEATURES], treino[TARGET]
X_test, y_test = teste[FEATURES], teste[TARGET]

print(f'Treino: {len(treino)} observações | {treino.data.min().date()} a {treino.data.max().date()}')
print(f'Teste:  {len(teste)} observações | {teste.data.min().date()} a {teste.data.max().date()}')


## 4. Modelos treinados

Foram escolhidos três modelos de naturezas diferentes:

1. **Ridge Regression (scikit-learn):** modelo linear regularizado usado como baseline.
2. **Gradient Boosting Regressor:** conjunto de árvores treinadas sequencialmente para corrigir os erros das anteriores, representando a família de árvores de decisão melhoradas por boosting.
3. **MLP Regressor profunda:** rede neural com três camadas ocultas (64, 32 e 16 neurônios). O escalonamento é feito dentro de um pipeline. A rede é usada mesmo com uma base pequena, atendendo ao requisito de comparar uma arquitetura neural profunda com os demais métodos.


In [ ]:
modelos = {
    'Ridge': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0))
    ]),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.03,
        max_depth=2,
        loss='huber',
        random_state=RANDOM_STATE
    ),
    'Rede Neural MLP': Pipeline([
        ('scaler', StandardScaler()),
        ('model', MLPRegressor(
            hidden_layer_sizes=(64, 32, 16),
            activation='relu',
            solver='adam',
            alpha=0.001,
            learning_rate_init=0.001,
            max_iter=5000,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=150,
            random_state=RANDOM_STATE
        ))
    ])
}

predicoes = {}
resultados = []

for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    predicoes[nome] = pred
    resultados.append({
        'Modelo': nome,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': mean_squared_error(y_test, pred) ** 0.5,
        'R2': r2_score(y_test, pred)
    })

resultados_df = pd.DataFrame(resultados).sort_values('RMSE').reset_index(drop=True)
display(resultados_df.round(4))


## 5. Avaliação e comparação

Usamos **MAE**, **RMSE** e **R²**. MAE e RMSE medem erro de previsão em pontos percentuais do IPCA mensal; quanto menores, melhor. O R² mede quanto da variação do alvo é explicada no conjunto de teste; quanto maior, melhor. Em séries econômicas pequenas e voláteis, um R² baixo ou negativo no teste é possível e deve ser reportado, não escondido.


In [ ]:
comparacao = pd.DataFrame({
    'data_previsao': teste['data'].reset_index(drop=True) + pd.offsets.MonthBegin(1),
    'real': y_test.reset_index(drop=True)
})
for nome, pred in predicoes.items():
    comparacao[nome] = pred

display(comparacao.round(3))

plt.figure(figsize=(12, 5))
plt.plot(comparacao['data_previsao'], comparacao['real'], marker='o', label='IPCA real')
for nome in modelos:
    plt.plot(comparacao['data_previsao'], comparacao[nome], marker='.', linestyle='--', label=nome)
plt.title('IPCA mensal — valor real x previsões dos modelos')
plt.ylabel('% no mês')
plt.xlabel('Mês previsto')
plt.grid(alpha=0.25)
plt.legend()
plt.show()


## 6. Interpretação final e relação com a Parte 1

A célula abaixo identifica automaticamente o modelo com menor RMSE no conjunto de teste e produz um resumo com as métricas observadas. A comparação deve ser interpretada junto aos indicadores da Parte 1: IPCA em 12 meses, Selic média mensal e defasagem entre os movimentos das séries.

Os KPIs da Parte 1 são **descritivos**, mostrando o que ocorreu. Os modelos deste checkpoint são **preditivos**, tentando estimar o IPCA do mês seguinte com base apenas em informações disponíveis até o mês corrente. Assim, as duas partes se complementam: a primeira descreve o ciclo econômico e a segunda testa até que ponto os padrões recentes contêm sinal útil para previsão de curto prazo.

Importante: o desempenho preditivo não prova que mudanças na Selic causam diretamente mudanças no IPCA. Política monetária, expectativas, câmbio, atividade econômica e choques externos também influenciam a inflação.


In [ ]:
melhor = resultados_df.iloc[0]
print('RESUMO FINAL')
print('-' * 60)
print(f"Melhor modelo pelo RMSE: {melhor['Modelo']}")
print(f"MAE:  {melhor['MAE']:.4f} p.p.")
print(f"RMSE: {melhor['RMSE']:.4f} p.p.")
print(f"R²:   {melhor['R2']:.4f}")
print()
print('Interpretação: o modelo acima apresentou o menor erro quadrático médio')
print('no período de teste. Ele deve ser lido como complemento aos indicadores')
print('históricos da Parte 1, fornecendo uma visão preditiva de curto prazo.')


## 7. Limitações

- O recorte 2020–2026 é curto para Machine Learning e contém poucos ciclos econômicos completos.
- A rede neural possui maior capacidade que o volume de dados disponível, portanto não é necessariamente o modelo mais adequado; ela foi incluída para a comparação exigida no checkpoint.
- A previsão é de curto prazo e usa apenas inflação e Selic, sem câmbio, atividade, expectativas ou preços administrados.
- Não há embaralhamento da série para preservar a ordem temporal.
- As conclusões são preditivas/associativas, não causais.
